In [27]:
import io
import zipfile
import requests
import frontmatter
from dotenv import load_dotenv
load_dotenv()


True

In [2]:
url = 'https://codeload.github.com/debauchee/barrier-wiki/faq/zip/refs/heads/main'
resp = requests.get(url)

In [3]:
def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.
    
    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name
    
    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com' 
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/master'
    resp = requests.get(url)
    
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    
    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md') 
            or filename_lower.endswith('.mdx')):
            continue
    
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    zf.close()
    return repository_data


In [4]:
barrier_docs = read_repo_data('debauchee', 'barrier-wiki')

In [5]:
print(f"barrier docs: {len(barrier_docs)}")

barrier docs: 10


In [6]:
def sliding_window(seq, size, step):
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        chunk = seq[i:i+size]
        result.append({'start': i, 'chunk': chunk})
        if i + size >= n:
            break

    return result

In [7]:
barrier_chunks = []

for doc in barrier_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    chunks = sliding_window(doc_content, 2000, 1000)
    for chunk in chunks:
        chunk.update(doc_copy)
    barrier_chunks.extend(chunks)

In [8]:
print(type(barrier_chunks[0]))
print(barrier_chunks[0].keys())

<class 'dict'>
dict_keys(['start', 'chunk', 'filename'])


In [9]:
from pprint import pprint
pprint(barrier_chunks[0]['chunk'][:200])

('## Adding Barrier to the Windows Firewall\n'
 '\n'
 'Screenshots captured by [l0o0](https://github.com/l0o0)\n'
 '\n'
 '![search windows '
 'defender](https://user-images.githubusercontent.com/2743744/49057950-ca40bd00-f23c-')


In [10]:
import re
text = barrier_docs[6]['content']
paragraphs = re.split(r"\n\s*\n", text.strip())

In [11]:
import re

def split_markdown_by_level(text, level=2):
    """
    Split markdown text by a specific header level.

    :param text: Markdown text as a string
    :param level: Header level to split on
    :return: List of sections as strings
    """
    # This regex matches markdown headers
    #For level 2, it matches lines starting with "## "
    header_pattern = r'^(#{' + str(level) + r'} )(.+)$'
    pattern = re.compile(header_pattern, re.MULTILINE)

    # Split and keep the headers
    parts = pattern.split(text)

    sections = []
    for i in range(1, len(parts), 3):
        # We step by 3 because regex.split() with
        # capturing groups returns:
        # [before_match, group1, group2, after_match, ...]
        # here group1 is "## ", group2 is the header text
        header = parts[i] + parts[i+1] # "##" + "Title"
        header = header.strip()

        # Get the content after this header
        content = ""
        if i+2 < len(parts):
            content = parts[i+2].strip()

        if content:
            section = f'{header}\n\n{content}'
        else:
            section = header
        sections.append(section)

    return sections
    

In [12]:
barrier_chunks = []

for doc in barrier_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    sections = split_markdown_by_level(doc_content, level=2)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        barrier_chunks.append(section_doc)

In [13]:
from minsearch import Index

barrier_index = Index(
    text_fields=["chunk", "title", "description", "filename"],
    keyword_fields=[]
)

barrier_index.fit(barrier_chunks)

In [14]:
print(barrier_chunks[0].keys())

dict_keys(['filename', 'section'])


In [15]:
barrier_faq = read_repo_data('debauchee', 'barrier-wiki')



In [16]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('multi-qa-distilbert-cos-v1')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [17]:
record = de_barrier_faq[2]
text = record['question'] + ' ' + record['content']
v_doc = embedding_model.encode(text)

NameError: name 'de_barrier_faq' is not defined

In [18]:
query = 'How do I install barrier on Mac OS?'
v_query = embedding_model.encode(query)

In [19]:
query = 'How do I install barrier on Mac OS?'
results = index.search(query)

NameError: name 'index' is not defined

In [21]:
v_doc = embedding_model.encode(text)

similarity = v_query.dot(v_doc)

In [22]:
from tqdm.auto import tqdm
from minsearch import VectorSearch
import numpy as np

barrier_embeddings = []

for d in tqdm(barrier_chunks):
    v = embedding_model.encode(d['section'])
    barrier_embeddings.append(v)

barrier_embeddings = np.array(barrier_embeddings)

np.save("barrier_embeddings.npy", barrier_embeddings)

barrier_vindex = VectorSearch()
barrier_vindex.fit(barrier_embeddings, barrier_chunks)

  0%|          | 0/37 [00:00<?, ?it/s]

In [23]:
def text_search(query):
    return barrier_index.search(query, num_results=5)

def vector_search(query):
    q = embedding_model.encode(query)
    return barrier_vindex.search(q, num_results=5)

def hybrid_search(query):
    text_results = text_search(query)
    vector_results = vector_search(query)

    # Combine and dedupe results
    seen_ids = set()
    combined_results = []

    for result in text_results + vector_results:
        if result['filename'] not in seen_ids:
            seen_ids.add(result['filename'])
            combined_results.append(result)

    return combined_results

In [ ]:
pprint(text_search('How do I install barrier in Mac OS?'))

In [ ]:
pprint(vector_search('How do I install barrier in Mac OS?'))

In [ ]:
pprint(hybrid_search('how do I install barrier in Linux?'))

In [25]:
from typing import List, Any

def text_search(query: str) -> List[Any]:
    """
    Perform a text-based search on the Barrier docs.

    Args:
        query(str): The search query string.

    Returns:
        List[Any]: A list of up to 5 search results returned from the Barrier docs.
    """
    return barrier_index.search(query, num_results=5)

In [29]:
system_prompt="You are a helpful agent that answers questions about Barrier using context from the Barrier documentation."

In [32]:
from pydantic_ai import Agent

agent = Agent(
    name="barrier_agent",
    instructions=system_prompt,
    tools=[text_search],
    model='openai-chat:gpt-4o-mini'
)

In [33]:
question = "Can I run Barrier on my Linux machine together with my Windows machine?"

result = await agent.run(user_prompt=question)

In [34]:
pprint(result)

AgentRunResult(output='Yes, you can run Barrier on both your Linux and Windows '
                      'machines. Barrier is designed to work across different '
                      'operating systems, allowing you to share your mouse and '
                      'keyboard between them seamlessly. \n'
                      '\n'
                      'Make sure to install Barrier on both systems; you can '
                      'follow the installation steps specific to each OS. For '
                      "Linux, you'll need to build it from source or install "
                      'it using your package manager, while for Windows, you '
                      'might want to use the installer or build it from source '
                      'as well. \n'
                      '\n'
                      'Once both systems have Barrier running, you can '
                      'configure them to work together. If you need more '
                      'detailed instructions on installation 

In [35]:
question = "Do I need to install bonjour to use Barrier?"
result = await agent.run(user_prompt=question)
pprint(result)

AgentRunResult(output='The search did not yield specific information regarding '
                      'the need to install Bonjour to use Barrier. However, '
                      'generally, Bonjour is not a mandatory requirement for '
                      'Barrier to function. \n'
                      '\n'
                      "If you're experiencing issues related to network "
                      'discovery or connections, installing Bonjour might help '
                      'in certain scenarios, especially when dealing with '
                      'devices on a local network. But for standard operation, '
                      'you should be able to run Barrier without it.\n'
                      '\n'
                      'If you have particular issues you are trying to '
                      'resolve, please provide more details, and I can assist '
                      'you further!')
